# Medical Image Segmentation with U-Net

This notebook adapts the ResNet50 classification pipeline into a full **semantic segmentation** workflow.  
Architecture: **U-Net** with a pretrained ResNet50 encoder.  
Task: Lung segmentation on chest X-rays (Montgomery/Shenzhen dataset or any binary mask dataset).  
XAI: Grad-CAM on the encoder + LIME superpixel visualisation adapted for segmentation.

In [ ]:
import os
import re
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import cv2
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import ResNet50
from sklearn.model_selection import train_test_split
from sklearn.metrics import jaccard_score

# Optional packages used later for XAI
# If Kaggle says lime is missing, uncomment the next line and run once:
# !pip -q install lime
import lime
from lime import lime_image
from skimage.segmentation import mark_boundaries

import warnings
warnings.filterwarnings('ignore')

IMG_SIZE   = 256
BATCH_SIZE = 8
EPOCHS     = 20
LR         = 1e-4

# Correct Kaggle dataset path
DATA_DIR = "/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation"

print("TensorFlow version:", tf.__version__)
print("DATA_DIR exists:", os.path.exists(DATA_DIR))
print("DATA_DIR contents:", os.listdir(DATA_DIR) if os.path.exists(DATA_DIR) else "Not found")

## 1. Dataset helpers — image + mask pairs

In [ ]:
def find_image_and_mask_dirs(data_dir):
    """Find the real image and mask folders for the Kaggle chest X-ray dataset."""
    possible_roots = [
        data_dir,
        os.path.join(data_dir, "Lung Segmentation"),
        "/kaggle/input/chest-xray-masks-and-labels/Lung Segmentation",
        "/kaggle/input/datasets/nikhilpandey360/chest-xray-masks-and-labels/Lung Segmentation",
    ]

    for root in possible_roots:
        img_dir = os.path.join(root, "CXR_png")
        mask_dir = os.path.join(root, "masks")
        if os.path.isdir(img_dir) and os.path.isdir(mask_dir):
            return img_dir, mask_dir

    raise FileNotFoundError(
        "Could not find CXR_png and masks folders. Check DATA_DIR. "
        f"Tried: {possible_roots}"
    )


def mask_candidates(image_name):
    """Return likely mask filenames for a given image filename."""
    stem, ext = os.path.splitext(image_name)
    return [
        image_name,
        stem + "_mask" + ext,
        stem + "_mask.png",
        stem.replace(".png", "") + "_mask.png",
    ]


def load_all_paths(data_dir):
    """Load all available image-mask pairs from CXR_png and masks."""
    img_dir, mask_dir = find_image_and_mask_dirs(data_dir)

    image_names = sorted([
        f for f in os.listdir(img_dir)
        if f.lower().endswith((".png", ".jpg", ".jpeg"))
    ])

    mask_files = set(os.listdir(mask_dir))
    imgs, masks, missing = [], [], []

    for image_name in image_names:
        matched_mask = None

        for cand in mask_candidates(image_name):
            if cand in mask_files:
                matched_mask = cand
                break

        if matched_mask is None:
            missing.append(image_name)
            continue

        imgs.append(os.path.join(img_dir, image_name))
        masks.append(os.path.join(mask_dir, matched_mask))

    if len(imgs) == 0:
        raise ValueError(
            "No image-mask pairs found. Inspect filenames in CXR_png and masks."
        )

    print(f"Image folder: {img_dir}")
    print(f"Mask folder : {mask_dir}")
    print(f"Total paired samples: {len(imgs)}")
    print(f"Images without masks skipped: {len(missing)}")

    if missing[:5]:
        print("Example missing masks:", missing[:5])

    return imgs, masks


def split_paths(imgs, masks, val_size=0.15, test_size=0.15, seed=42):
    """Create train/validation/test splits from paired paths."""
    train_imgs, test_imgs, train_masks, test_masks = train_test_split(
        imgs,
        masks,
        test_size=test_size,
        random_state=seed,
        shuffle=True
    )

    val_relative_size = val_size / (1.0 - test_size)

    train_imgs, val_imgs, train_masks, val_masks = train_test_split(
        train_imgs,
        train_masks,
        test_size=val_relative_size,
        random_state=seed,
        shuffle=True
    )

    print(f"Train samples: {len(train_imgs)}")
    print(f"Val samples  : {len(val_imgs)}")
    print(f"Test samples : {len(test_imgs)}")

    return train_imgs, train_masks, val_imgs, val_masks, test_imgs, test_masks


def parse_image_mask(img_path, mask_path):
    """Load, resize, normalize image and binarize mask."""
    img = tf.io.read_file(img_path)
    img = tf.image.decode_png(img, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    img = tf.cast(img, tf.float32) / 255.0

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    mask = tf.image.resize(mask, [IMG_SIZE, IMG_SIZE], method="nearest")
    mask = tf.cast(mask > 127, tf.float32)

    return img, mask


@tf.function
def augment(img, mask):
    """Apply identical spatial augmentation to image and mask."""
    seed = tf.random.uniform(shape=[2], maxval=10000, dtype=tf.int32)

    img = tf.image.stateless_random_flip_left_right(img, seed=seed)
    mask = tf.image.stateless_random_flip_left_right(mask, seed=seed)

    k = tf.random.uniform((), 0, 4, dtype=tf.int32)
    img = tf.image.rot90(img, k)
    mask = tf.image.rot90(mask, k)

    img = tf.image.random_brightness(img, 0.1)
    img = tf.clip_by_value(img, 0.0, 1.0)

    return img, mask


def make_dataset(imgs, masks, augment_data=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((imgs, masks))
    ds = ds.map(parse_image_mask, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(
            buffer_size=len(imgs),
            seed=42,
            reshuffle_each_iteration=True
        )

    if augment_data:
        ds = ds.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

    return ds


def build_generators(data_dir):
    imgs, masks = load_all_paths(data_dir)

    train_imgs, train_masks, val_imgs, val_masks, test_imgs, test_masks = split_paths(
        imgs,
        masks
    )

    train_ds = make_dataset(
        train_imgs,
        train_masks,
        augment_data=True,
        shuffle=True
    )

    val_ds = make_dataset(
        val_imgs,
        val_masks
    )

    test_ds = make_dataset(
        test_imgs,
        test_masks
    )

    return train_ds, val_ds, test_ds, (
        train_imgs,
        train_masks,
        val_imgs,
        val_masks,
        test_imgs,
        test_masks
    )


train_ds, val_ds, test_ds, path_splits = build_generators(DATA_DIR)

# Quick sanity check
for sample_imgs, sample_masks in train_ds.take(1):
    print("Image batch shape:", sample_imgs.shape)
    print("Mask batch shape :", sample_masks.shape)
    print("Image range      :", float(tf.reduce_min(sample_imgs)), "to", float(tf.reduce_max(sample_imgs)))
    print("Mask values      :", np.unique(sample_masks.numpy()))

## 2. U-Net with ResNet50 encoder

In [ ]:
def conv_block(x, filters, kernel=3):
    """Two Conv-BN-ReLU layers."""
    x = layers.Conv2D(filters, kernel, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    x = layers.Conv2D(filters, kernel, padding='same', use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('relu')(x)
    return x


def upsample_block(x, skip, filters):
    """Transpose-conv upsample + skip connection + conv block."""
    x = layers.Conv2DTranspose(filters, 2, strides=2, padding='same')(x)
    x = layers.Concatenate()([x, skip])
    x = conv_block(x, filters)
    return x


def build_unet_resnet50(img_size=256, num_classes=1):
    """
    U-Net with a frozen ResNet50 encoder.
    Skip connections from: conv1_relu, conv2_block3_1_relu,
                           conv3_block4_1_relu, conv4_block6_1_relu
    Bottleneck: conv5_block3_out
    """
    base = ResNet50(weights='imagenet', include_top=False,
                    input_shape=(img_size, img_size, 3))
    base.trainable = False

    # ---- Encoder skip layers ----
    s1 = base.get_layer('conv1_relu').output          # 128x128, 64
    s2 = base.get_layer('conv2_block3_1_relu').output # 64x64,  256
    s3 = base.get_layer('conv3_block4_1_relu').output # 32x32,  512
    s4 = base.get_layer('conv4_block6_1_relu').output # 16x16, 1024
    b  = base.get_layer('conv5_block3_out').output    # 8x8,   2048  (bottleneck)

    # ---- Decoder ----
    x = conv_block(b, 1024)
    x = upsample_block(x, s4,  512)
    x = upsample_block(x, s3,  256)
    x = upsample_block(x, s2,  128)
    x = upsample_block(x, s1,   64)

    # Final upsample to original size + pixel-wise output
    x = layers.Conv2DTranspose(32, 2, strides=2, padding='same')(x)
    x = conv_block(x, 32)

    if num_classes == 1:
        outputs = layers.Conv2D(1, 1, activation='sigmoid')(x)
    else:
        outputs = layers.Conv2D(num_classes, 1, activation='softmax')(x)

    model = models.Model(inputs=base.input, outputs=outputs, name='UNet_ResNet50')
    return model


model = build_unet_resnet50(IMG_SIZE)
model.summary()

## 3. Loss functions — Dice + BCE combo

In [ ]:
def dice_loss(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return 1 - (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def bce_dice_loss(y_true, y_pred):
    bce  = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    dice = dice_loss(y_true, y_pred)
    return bce + dice

def dice_coeff(y_true, y_pred, smooth=1e-6):
    """Metric version of Dice (higher = better)."""
    y_pred_b = tf.cast(y_pred > 0.5, tf.float32)
    y_true_f = tf.reshape(y_true,   [-1])
    y_pred_f = tf.reshape(y_pred_b, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou_metric(y_true, y_pred, smooth=1e-6):
    """Intersection over Union."""
    y_pred_b = tf.cast(y_pred > 0.5, tf.float32)
    y_true_f = tf.reshape(y_true,   [-1])
    y_pred_f = tf.reshape(y_pred_b, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


model.compile(
    optimizer=optimizers.Adam(LR),
    loss=bce_dice_loss,
    metrics=[dice_coeff, iou_metric]
)

## 4. Training

In [ ]:
def train_model(model, train_ds, val_ds):
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            'best_seg_model.keras',
            save_best_only=True,
            monitor='val_dice_coeff',
            mode='max'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', patience=3, factor=0.5, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_dice_coeff', patience=7, mode='max', restore_best_weights=True
        )
    ]
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks
    )
    return history

history = train_model(model, train_ds, val_ds)

## 5. Training curves

In [ ]:
def plot_training(history):
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    pairs  = [
        ('loss',       'val_loss',       'BCE + Dice Loss'),
        ('dice_coeff', 'val_dice_coeff', 'Dice Coefficient'),
        ('iou_metric', 'val_iou_metric', 'IoU')
    ]
    for ax, (train_m, val_m, title) in zip(axes, pairs):
        ax.plot(history.history[train_m], label='Train')
        ax.plot(history.history[val_m],   label='Val')
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.legend()
    plt.tight_layout()
    plt.show()

plot_training(history)

## 6. Evaluation — pixel-level metrics

In [ ]:
def evaluate_model(model, test_ds):
    model.load_weights('best_seg_model.keras')

    all_dice, all_iou = [], []
    all_preds, all_masks = [], []

    for imgs, masks in test_ds:
        preds = model.predict(imgs, verbose=0)
        preds_b = (preds > 0.5).astype(np.float32)

        for p, m in zip(preds_b, masks.numpy()):
            p_flat = p.flatten()
            m_flat = m.flatten()
            intersection = np.sum(p_flat * m_flat)
            dice = (2 * intersection + 1e-6) / (np.sum(p_flat) + np.sum(m_flat) + 1e-6)
            union = np.sum(p_flat) + np.sum(m_flat) - intersection
            iou  = (intersection + 1e-6) / (union + 1e-6)
            all_dice.append(dice)
            all_iou.append(iou)
            all_preds.append(p_flat)
            all_masks.append(m_flat)

    all_preds_cat = np.concatenate(all_preds).astype(int)
    all_masks_cat = np.concatenate(all_masks).astype(int)
    pixel_acc     = np.mean(all_preds_cat == all_masks_cat)

    print(f"Mean Dice Coefficient : {np.mean(all_dice):.4f} ± {np.std(all_dice):.4f}")
    print(f"Mean IoU              : {np.mean(all_iou):.4f} ± {np.std(all_iou):.4f}")
    print(f"Pixel Accuracy        : {pixel_acc:.4f}")

    # Distribution plots
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(all_dice, bins=30, edgecolor='k', color='steelblue')
    axes[0].set_title('Dice Score Distribution')
    axes[0].set_xlabel('Dice')
    axes[1].hist(all_iou,  bins=30, edgecolor='k', color='coral')
    axes[1].set_title('IoU Distribution')
    axes[1].set_xlabel('IoU')
    plt.tight_layout()
    plt.show()

    return np.array(all_dice), np.array(all_iou)

dice_scores, iou_scores = evaluate_model(model, test_ds)

## 7. Visual prediction samples

In [ ]:
def visualize_predictions(model, test_ds, num_samples=4):
    model.load_weights('best_seg_model.keras')
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    fig.suptitle('Segmentation Results', fontsize=16, fontweight='bold')

    for imgs, masks in test_ds.take(1):
        preds = model.predict(imgs, verbose=0)

        for i in range(min(num_samples, len(imgs))):
            img    = imgs[i].numpy()
            mask   = masks[i].numpy().squeeze()
            pred   = preds[i].squeeze()
            pred_b = (pred > 0.5).astype(np.float32)

            # Overlay: green = True Positive, red = FP, blue = FN
            overlay = img.copy()
            tp = (pred_b == 1) & (mask == 1)
            fp = (pred_b == 1) & (mask == 0)
            fn = (pred_b == 0) & (mask == 1)
            overlay[tp] = [0, 0.8, 0]
            overlay[fp] = [0.8, 0, 0]
            overlay[fn] = [0, 0, 0.8]

            dice = (2 * np.sum(pred_b * mask) + 1e-6) / (np.sum(pred_b) + np.sum(mask) + 1e-6)

            axes[i, 0].imshow(img)
            axes[i, 0].set_title('Input Image')
            axes[i, 1].imshow(mask, cmap='gray')
            axes[i, 1].set_title('Ground Truth')
            axes[i, 2].imshow(pred, cmap='hot')
            axes[i, 2].set_title('Predicted Prob Map')
            axes[i, 3].imshow(overlay)
            axes[i, 3].set_title(f'Overlay | Dice={dice:.3f}\nGreen=TP, Red=FP, Blue=FN')

            for ax in axes[i]:
                ax.axis('off')

    plt.tight_layout()
    plt.savefig('seg_predictions.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_predictions(model, test_ds)

## 8. XAI — Grad-CAM adapted for segmentation

For segmentation we compute Grad-CAM w.r.t. the **mean predicted mask probability** rather than a class logit.

In [ ]:
def make_gradcam_seg(img_array, model, last_conv_layer='conv4_block6_out'):
    """
    Grad-CAM for segmentation: gradient of mean(output) w.r.t. a deep encoder layer.
    Returns a heatmap resized to (IMG_SIZE, IMG_SIZE).
    """
    # Build a sub-model that outputs (target_conv, full_prediction)
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer).output, model.output]
    )

    with tf.GradientTape() as tape:
        conv_out, pred = grad_model(img_array, training=False)
        # Use mean predicted probability as scalar loss
        loss = tf.reduce_mean(pred)

    grads       = tape.gradient(loss, conv_out)           # (1, h, w, C)
    pooled      = tf.reduce_mean(grads, axis=(1, 2))[0]   # (C,)
    heatmap     = tf.reduce_sum(conv_out[0] * pooled, axis=-1)  # (h, w)
    heatmap     = tf.maximum(heatmap, 0)
    heatmap     = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    heatmap     = heatmap.numpy()
    return cv2.resize(heatmap, (IMG_SIZE, IMG_SIZE))


def overlay_gradcam(img, heatmap, alpha=0.4):
    colormap  = cm.get_cmap('jet')
    heatmap_c = colormap(heatmap)[:, :, :3]
    return (img * (1 - alpha) + heatmap_c * alpha).clip(0, 1)


def visualize_gradcam_seg(model, test_ds, num_samples=4):
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, num_samples * 4))
    fig.suptitle('Grad-CAM Analysis (Segmentation)', fontsize=16, fontweight='bold')

    for imgs, masks in test_ds.take(1):
        for i in range(min(num_samples, len(imgs))):
            img      = imgs[i].numpy()
            mask     = masks[i].numpy().squeeze()
            img_arr  = np.expand_dims(img, 0)

            pred     = model.predict(img_arr, verbose=0)[0].squeeze()
            heatmap  = make_gradcam_seg(img_arr, model)
            overlay  = overlay_gradcam(img, heatmap)

            axes[i, 0].imshow(img)
            axes[i, 0].set_title('Input Image')
            axes[i, 1].imshow(mask, cmap='gray')
            axes[i, 1].set_title('Ground Truth Mask')
            axes[i, 2].imshow(heatmap, cmap='jet')
            axes[i, 2].set_title('Grad-CAM Heatmap')
            axes[i, 3].imshow(overlay)
            axes[i, 3].set_title('Grad-CAM Overlay')

            for ax in axes[i]:
                ax.axis('off')

    plt.tight_layout()
    plt.savefig('gradcam_seg.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_gradcam_seg(model, test_ds)

## 9. XAI — LIME adapted for segmentation

LIME perturbs superpixels and measures the change in **mean predicted mask probability** to rank which image regions drive the segmentation output.

In [ ]:
def lime_seg_predict_fn(images):
    """
    LIME expects shape (N, H, W, 3) uint8 in [0,255].
    We return a 2-column probability matrix where column 1 = mean predicted mask.
    """
    images_f = images.astype(np.float32) / 255.0
    preds    = model.predict(images_f, verbose=0)  # (N, H, W, 1)
    mean_seg = preds.mean(axis=(1, 2, 3))          # (N,)  mean mask probability
    return np.column_stack([1 - mean_seg, mean_seg])


def visualize_lime_seg(model, test_ds, num_samples=4):
    explainer = lime_image.LimeImageExplainer()

    fig, axes = plt.subplots(num_samples, 3, figsize=(12, num_samples * 4))
    fig.suptitle('LIME Analysis (Segmentation)', fontsize=16, fontweight='bold')

    for imgs, masks in test_ds.take(1):
        for i in range(min(num_samples, len(imgs))):
            img      = imgs[i].numpy()
            mask     = masks[i].numpy().squeeze()
            img_uint = (img * 255).astype(np.uint8)

            explanation = explainer.explain_instance(
                img_uint, lime_seg_predict_fn,
                top_labels=2, hide_color=0, num_samples=500
            )

            # Use label=1 ("segmentation present" direction)
            temp_all, mask_all = explanation.get_image_and_mask(
                1, positive_only=False, num_features=10, hide_rest=False
            )
            temp_pos, mask_pos = explanation.get_image_and_mask(
                1, positive_only=True,  num_features=5,  hide_rest=True
            )

            axes[i, 0].imshow(img)
            axes[i, 0].set_title('Input Image')
            axes[i, 0].axis('off')

            axes[i, 1].imshow(mark_boundaries(temp_all / 255.0, mask_all))
            axes[i, 1].set_title('LIME All Superpixels')
            axes[i, 1].axis('off')

            axes[i, 2].imshow(mark_boundaries(temp_pos / 255.0, mask_pos))
            axes[i, 2].set_title('LIME Top-5 Features')
            axes[i, 2].axis('off')

    plt.tight_layout()
    plt.savefig('lime_seg.png', dpi=150, bbox_inches='tight')
    plt.show()

visualize_lime_seg(model, test_ds)

## 10. Full XAI dashboard

In [ ]:
def xai_dashboard(model, test_ds, num_samples=3):
    explainer = lime_image.LimeImageExplainer()

    fig, axes = plt.subplots(num_samples, 5, figsize=(20, num_samples * 4))
    fig.suptitle(
        'XAI Dashboard: Image | GT Mask | Prediction | Grad-CAM | LIME',
        fontsize=14, fontweight='bold'
    )

    for imgs, masks in test_ds.take(1):
        for i in range(min(num_samples, len(imgs))):
            img      = imgs[i].numpy()
            mask     = masks[i].numpy().squeeze()
            img_uint = (img * 255).astype(np.uint8)
            img_arr  = np.expand_dims(img, 0)

            pred    = model.predict(img_arr, verbose=0)[0].squeeze()
            pred_b  = (pred > 0.5).astype(np.float32)
            dice    = (2 * np.sum(pred_b * mask) + 1e-6) / (np.sum(pred_b) + np.sum(mask) + 1e-6)

            heatmap = make_gradcam_seg(img_arr, model)
            overlay = overlay_gradcam(img, heatmap)

            explanation = explainer.explain_instance(
                img_uint, lime_seg_predict_fn,
                top_labels=2, hide_color=0, num_samples=300
            )
            temp, lime_mask = explanation.get_image_and_mask(
                1, positive_only=True, num_features=5, hide_rest=True
            )

            axes[i, 0].imshow(img);                                    axes[i, 0].set_title('Input')
            axes[i, 1].imshow(mask, cmap='gray');                      axes[i, 1].set_title('GT Mask')
            axes[i, 2].imshow(pred, cmap='hot');                       axes[i, 2].set_title(f'Pred | Dice={dice:.3f}')
            axes[i, 3].imshow(overlay);                                axes[i, 3].set_title('Grad-CAM')
            axes[i, 4].imshow(mark_boundaries(temp / 255.0, lime_mask)); axes[i, 4].set_title('LIME')

            for ax in axes[i]:
                ax.axis('off')

    plt.tight_layout()
    plt.savefig('xai_dashboard_seg.png', dpi=150, bbox_inches='tight')
    plt.show()

xai_dashboard(model, test_ds)